In [12]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-google-genai \
langchain-huggingface \
langchain-chroma \
chromadb \
sentence-transformers \
pypdf \
python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\asus\AppData\Local\Temp\ipykernel_20144\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"

print("API Key Loaded:", os.environ["GOOGLE_API_KEY"][:8] + "...")

API Key Loaded: YOUR_GOO...


In [4]:
from pathlib import Path

pdf_path = Path("RAG_Chatbot_Streamlit/researchpaper.pdf")

print("PDF exists:", pdf_path.exists())
print("PDF path:", pdf_path.resolve())

PDF exists: True
PDF path: C:\Users\asus\Downloads\ResearchPaperBot\ResearchPaperBot\RAG_Chatbot_Streamlit\researchpaper.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "C:/Users/asus/Downloads/ResearchPaperBot/ResearchPaperBot/RAG_Chatbot_Streamlit/researchpaper.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"✅ PDF Loaded Successfully!")
print(f"Pages Loaded: {len(documents)}")

✅ PDF Loaded Successfully!
Pages Loaded: 41


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")
print(chunks[0].page_content[:500])

Total Chunks: 176
Chapter inAdvancing Astrophysics with the SKA – II
Probing Neutron Star Interiors and the Properties of Cold
Ultra-dense Matter with the SKAO
A. Basu,1 V. Graber,2 M. E. Lower,3 M. Antonelli,4 D. Antonopoulou,1 M. Bagchi,5,6 P.
Char,7 P. C. C. Freire,8 B. Haskell,9,10,11 H. Hu,12,8 D. I. Jones,13 B. Mukhopadhyay,14 M.
Oertel,15 N. Rea,16,17 V. Sagun,13 B. Shaw,1 J. Singha,18 B. W. Stappers,1 T.
Thongmeearkom,19,1 A. L. Watts,20 P. Weltevrede1 and The SKA Pulsar Science Working
Group
1Jodrell Ban


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

C:\Users\asus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 573.78it/s]


In [10]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Chroma Vector Database Created Successfully!")
print(f"Stored {len(chunks)} chunks.")

✅ Chroma Vector Database Created Successfully!
Stored 176 chunks.


In [11]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Retriever is ready!")


✅ Retriever is ready!


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

print("✅ Gemini initialized!")

✅ Gemini initialized!


In [13]:
def ask(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful AI assistant.

Answer the question ONLY using the context below.
If the answer is not found in the context, reply:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    return response.content

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

print("✅ Gemini initialized successfully!")

✅ Gemini initialized successfully!


In [15]:
def ask(question):
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful AI assistant.

Answer the question ONLY using the context below.
If the answer is not found in the context, reply:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)
    return response.content

In [16]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Available models:\n")

for model in client.models.list():
    print(model.name)

Available models:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyri

In [17]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Available Models:\n")

for model in client.models.list():
    print(model.name)

Available Models:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyri

In [18]:
model="models/gemini-2.5-flash"

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

print("✅ Gemini initialized successfully!")

✅ Gemini initialized successfully!


In [20]:
import langchain_google_genai
import google.genai

print("LangChain Google GenAI:", langchain_google_genai.__version__)
print("Google GenAI:", google.genai.__version__)

LangChain Google GenAI: 4.4.0
Google GenAI: 2.28.0


In [21]:
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"

In [22]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini

In [23]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Hello"
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello! How can I help you today?


In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

print("✅ LLM initialized!")

✅ LLM initialized!


In [25]:
response = llm.invoke("Say Hello")

print(response.content)

C:\Users\asus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TKxj546bfA00GF1S6kB2x92LuBQ79BOzM6HtIlma6tFKLxXkKhMpdZb1Y6P3F/JA92BHxif+9mdyWOGTyiy9IhzM899phG6HLYGez+WmtKn+11+XldQsFz2c='}}]


In [26]:
def ask(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful AI assistant.

Answer the question only using the context below.
If the answer is not present in the context, say:
"I couldn't find the answer in the document."

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    # Handle both string and structured responses
    if isinstance(response.content, str):
        return response.content

    return "".join(
        item.get("text", "")
        for item in response.content
        if isinstance(item, dict)
    )

In [28]:
question = "Why is constraining the equation of state (EoS) essential for understanding neutron star interiors?"

answer = ask(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

C:\Users\asus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question:
Why is constraining the equation of state (EoS) essential for understanding neutron star interiors?

Answer:
Based on the provided context, constraining the equation of state (EoS) is essential to inferring key aspects of dense matter microphysics.


In [29]:
docs = retriever.invoke("What is the main objective of this research paper?")

print("Retrieved Documents:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n----- Document {i+1} -----")
    print(doc.page_content[:500])

Retrieved Documents: 3

----- Document 1 -----
M. Ge et al.arXiv e-prints, art. arXiv:2211.03246, Nov. 2022. doi: 10.48550/arXiv.2211.03246.
M. Y. Ge et al.ApJ, 896(1):55, June 2020. doi: 10.3847/1538-4357/ab8db6.
K.C.Gendreauetal. InJ.-W.A.denHerder,T.Takahashi,andM.Bautz,editors,SpaceTelescopes
andInstrumentation2016: UltraviolettoGammaRay,volume9905ofSocietyofPhoto-Optical
Instrumentation Engineers (SPIE) Conference Series, page 99051H, July 2016. doi: 10.1117/
12.2231304.
M. Geyer et al.A&A, 674:A169, June 2023. doi: 10.1051/0004-6361/20

----- Document 2 -----
N. Martin and M. Urban.Phys. Rev. C, 94(6):065801, Dec. 2016. doi: 10.1103/PhysRevC.94.
065801.
J. G. Martinez et al.ApJ, 812(2):143, Oct. 2015. doi: 10.1088/0004-637X/812/2/143.
A. Melatos, C. Peralta, and J. S. B. Wyithe.ApJ, 672(2):1103–1118, Jan. 2008. doi: 10.1086/
523349.
A. Melatos, G. Howitt, and W. Fulgenzi.ApJ, 863(2):196, Aug. 2018. doi: 10.3847/1538-4357/
aad228.
37

----- Document 3 -----
M. E. Lower et al.MNRAS

In [30]:
%who

ChatGoogleGenerativeAI	 Chroma	 HuggingFaceEmbeddings	 Path	 PyPDFLoader	 RecursiveCharacterTextSplitter	 answer	 ask	 chunks	 
client	 doc	 docs	 documents	 embeddings	 genai	 google	 i	 langchain_google_genai	 
llm	 loader	 m	 model	 os	 pdf_path	 question	 response	 retriever	 
text_splitter	 vectorstore	 


In [31]:
print(vectorstore._collection.count())

176


In [32]:
docs = retriever.invoke("What is the main objective of this research paper?")

print("Retrieved:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n----- Chunk {i+1} -----")
    print(doc.page_content[:500])

Retrieved: 3

----- Chunk 1 -----
M. Ge et al.arXiv e-prints, art. arXiv:2211.03246, Nov. 2022. doi: 10.48550/arXiv.2211.03246.
M. Y. Ge et al.ApJ, 896(1):55, June 2020. doi: 10.3847/1538-4357/ab8db6.
K.C.Gendreauetal. InJ.-W.A.denHerder,T.Takahashi,andM.Bautz,editors,SpaceTelescopes
andInstrumentation2016: UltraviolettoGammaRay,volume9905ofSocietyofPhoto-Optical
Instrumentation Engineers (SPIE) Conference Series, page 99051H, July 2016. doi: 10.1117/
12.2231304.
M. Geyer et al.A&A, 674:A169, June 2023. doi: 10.1051/0004-6361/20

----- Chunk 2 -----
N. Martin and M. Urban.Phys. Rev. C, 94(6):065801, Dec. 2016. doi: 10.1103/PhysRevC.94.
065801.
J. G. Martinez et al.ApJ, 812(2):143, Oct. 2015. doi: 10.1088/0004-637X/812/2/143.
A. Melatos, C. Peralta, and J. S. B. Wyithe.ApJ, 672(2):1103–1118, Jan. 2008. doi: 10.1086/
523349.
A. Melatos, G. Howitt, and W. Fulgenzi.ApJ, 863(2):196, Aug. 2018. doi: 10.3847/1538-4357/
aad228.
37

----- Chunk 3 -----
M. E. Lower et al.MNRAS, 508(3):3251–3274,

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

In [ ]:
docs = retriever.invoke("What is the main objective of this research paper?")

print("Number of documents:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n----- Document {i+1} -----")
    print(doc.page_content[:1000])

In [ ]:
for i, chunk in enumerate(chunks):
    text = chunk.page_content.lower()

    if "abstract" in text:
        print("="*80)
        print("Chunk:", i)
        print(chunk.page_content[:1500])
        break

In [ ]:
for i, chunk in enumerate(chunks):
    text = chunk.page_content.lower()

    if "introduction" in text:
        print("="*80)
        print("Chunk:", i)
        print(chunk.page_content[:1500])
        break

In [ ]:
from langchain_chroma import Chroma

# Delete the old vector database from memory
try:
    del vectorstore
except:
    pass

# Create a fresh vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector DB rebuilt!")
print("Documents:", vectorstore._collection.count())

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

print("Retriever ready!")

In [ ]:
docs = retriever.invoke("What is the main objective of this research paper?")

for i, doc in enumerate(docs):
    print("=" * 80)
    print(f"Document {i+1}")
    print(doc.page_content[:800])

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 50,
        "lambda_mult": 0.2
    }
)

In [ ]:
docs = retriever.invoke("Introduction")

for i, doc in enumerate(docs):
    print("="*80)
    print(doc.page_content[:800])

In [ ]:
results = vectorstore.similarity_search(
    "Introduction",
    k=5
)

for i, r in enumerate(results):
    print("="*80)
    print(f"Result {i+1}")
    print(r.page_content[:800])

In [ ]:
from langchain_chroma import Chroma
import shutil
import os

# Delete the old database if it exists
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

print("Old Chroma database deleted.")

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nCan write here?:", os.access(os.getcwd(), os.W_OK))

print("\nDirectory contents:")
print(os.listdir())

In [ ]:
import chromadb
print(chromadb.__version__)

In [ ]:
import sys
print(sys.version)

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="test_collection_2026"
)

print("Success!")
print(vectorstore._collection.count())

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 50,
        "lambda_mult": 0.2
    }
)

print("✅ Retriever ready!")

In [ ]:
docs = retriever.invoke("What is the main objective of this research paper?")

for i, doc in enumerate(docs):
    print("=" * 80)
    print(f"Document {i+1}")
    print(doc.page_content[:800])

In [ ]:
for i in range(10):
    print("=" * 80)
    print(f"Chunk {i}")
    print(chunks[i].page_content[:600])

In [ ]:
print(text_splitter)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

In [ ]:
for i in range(10):
    print("=" * 80)
    print(f"Chunk {i}")
    print(chunks[i].page_content[:600])

In [ ]:
question = "What is the main objective of this research paper?"

docs = retriever.invoke(question)

for i, d in enumerate(docs):
    print("=" * 80)
    print(f"Document {i+1}")
    print(d.page_content[:1000])

In [ ]:
!pip install sentence-transformers

In [ ]:
def ask(question):
    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful research assistant.

Answer ONLY using the context below.
If the answer is not available in the context, reply:
"I couldn't find the answer in the document."

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)
    return response.content

In [33]:
print(ask("What is the main objective of this paper?"))

C:\Users\asus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I couldn't find the answer in the document.


In [ ]:
docs = retriever.invoke("What is the main objective of this paper?")

print("Number of documents:", len(docs))

for i, doc in enumerate(docs):
    print("="*80)
    print(f"Document {i+1}")
    print(doc.page_content[:500])

NameError: name 'retriever' is not defined

In [ ]:
docs = retriever.invoke("What is the main objective of this paper?")

context = "\n\n".join([doc.page_content for doc in docs])

print(context[:5000])

In [ ]:
import os

def ask(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a research paper assistant.

Answer the question ONLY using the retrieved context.

If the answer is not present in the context, reply:
"I couldn't find the answer in the document."

Retrieved Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    print("\n==============================")
    print("Answer")
    print("==============================")

    # Extract Gemini response text
    if isinstance(response.content, list):
        answer = "".join(
            item.get("text", "")
            for item in response.content
            if isinstance(item, dict)
        )
    else:
        answer = response.content

    print(answer)

    print("\n==============================")
    print("Retrieved Sources")
    print("==============================")

    seen = set()

    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "Unknown"))
        page = doc.metadata.get("page", "Unknown")

        key = (source, page)

        if key not in seen:
            seen.add(key)
            print(f"📄 {source}  |  Page {page + 1}")

In [ ]:
docs = retriever.invoke("What is the main objective of this paper?")

print("Retrieved", len(docs), "documents\n")

for i, doc in enumerate(docs):
    print("=" * 80)
    print(f"Document {i+1}")
    print(doc.page_content[:1000])

# ==========================================
# Embedding Model Comparison
# ==========================================

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

mini_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ MiniLM Embedding Loaded")

In [ ]:
from langchain_chroma import Chroma

mini_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=mini_embeddings,
    collection_name="minilm_vectorstore"
)

print("Documents:", mini_vectorstore._collection.count())

In [ ]:
mini_retriever = mini_vectorstore.as_retriever(
    search_kwargs={"k":5}
)

print("Retriever Ready")

In [ ]:
docs = mini_retriever.invoke(
    "What is the main objective of this paper?"
)

for i, doc in enumerate(docs):
    print("="*80)
    print(f"Document {i+1}")
    print(doc.page_content[:500])

# Embedding Model Comparison

## Model 1
- BAAI/bge-base-en-v1.5

## Model 2
- sentence-transformers/all-MiniLM-L6-v2

## Comparison

| Feature | BGE Base | MiniLM |
|---------|----------|---------|
| Embedding Dimension | 768 | 384 |
| Speed | Medium | Fast |
| Memory Usage | Higher | Lower |
| Retrieval on Test PDF | Retrieved reference sections | Retrieved reference sections |

## Selected Model

BAAI/bge-base-en-v1.5 was selected because it generally provides richer semantic embeddings, although both models showed similar behavior on this complex scientific PDF.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

# Retrieval Strategy Comparison

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

In [ ]:
docs = mmr_retriever.invoke("What is the main objective of this paper?")

for i, doc in enumerate(docs):
    print("=" * 80)
    print(f"Document {i+1}")
    print(doc.page_content[:500])

# Retrieval Strategy Comparison

## Strategy 1
Similarity Search

### Observation
- Retrieved mostly reference sections.
- High similarity caused repeated reference pages.

## Strategy 2
MMR (Maximal Marginal Relevance)

### Observation
- Retrieved more diverse chunks.
- Included content from the paper along with references.
- Reduced duplicate information.

## Selected Strategy

MMR was selected because it provides greater diversity in retrieved documents, improving the context supplied to the language model.

In [ ]:
docs = retriever.invoke("What is the main objective of this paper?")

for i, doc in enumerate(docs):
    print("=" * 60)
    print(f"Document {i+1}")
    print(doc.metadata)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

In [ ]:
ask("What is the main objective of this paper?")

# RAG System Evaluation

The chatbot was evaluated using ten questions related to the uploaded research paper. Each response was assessed based on retrieval quality, answer correctness, and overall performance.

In [ ]:
questions = [
    "What is the main objective of this paper?",
    "What is SKAO?",
    "What are neutron stars?",
    "What is the equation of state?",
    "Why are pulsars important?",
    "What observational techniques are discussed?",
    "What future work is mentioned?",
    "Who are the authors of the paper?",
    "How many pages does the paper contain?",
    "Summarize the conclusion of the paper."
]

for i, q in enumerate(questions, 1):
    print("=" * 80)
    print(f"Question {i}: {q}")
    ask(q)
    print("\n")

Question 1: What is the main objective of this paper?


NameError: name 'ask' is not defined

## Overall Evaluation

The RAG chatbot successfully answered domain-specific questions when the retriever returned relevant document chunks. Questions related to neutron stars, pulsars, observational techniques, and the equation of state were answered accurately.

However, questions requiring information from the introduction, conclusion, or document metadata (such as authors and total pages) were not answered correctly because the required context was not retrieved. This demonstrates that the overall performance of the chatbot is highly dependent on retrieval quality.

The MMR retrieval strategy produced more diverse and relevant document chunks than standard similarity search and was therefore selected as the final retrieval method for the project.

In [ ]:
print("swetha")

swetha
